In [ ]:
!pip install -q -U "transformers>=4.45" accelerate bitsandbytes sentence-transformers faiss-cpu datasets huggingface_hub scipy

In [ ]:
# ============================== CONFIG (edit this cell only) ==============================
MODEL_KEY   = "llama1b"     # llama1b | llama3b | qwen3b | llama8b | mistral7b
DATASET_KEY = "squad"       # squad | nq
SEEDS       = [0, 1, 2, 3, 4]
OUT_ROOT    = "/kaggle/working/rlar"

# Data (DATA_SEED fixed across all models so every model sees identical questions)
N_TRAIN, N_VAL, N_TEST = 2000, 400, 600
DATA_SEED      = 2026
GROUP_FRACS    = dict(train=0.70, val=0.10, test=0.20)   # fraction of articles per split
PASSAGE_WORDS  = 100      # SQuAD contexts chunked to 100-word passages (DPR convention)
NQ_NEGS_PER_Q  = 10       # BM25 hard negatives kept per NQ question in the corpus

# Retrieval / generation
K_OPTIONS      = [0, 1, 2, 3]
TOPN_RETRIEVE  = 10       # cached per query; k-contexts are prefixes; the rest feeds Recall/MRR/nDCG
MAX_NEW_TOKENS = 32
GEN_BATCH      = {0: 32, 1: 24, 2: 16, 3: 12}   # auto-halves on OOM
GEN_CHUNK      = 256      # queries per checkpoint write
LATENCY_N      = 300      # test queries profiled at batch size 1

# Reward: r(q,k) = F1 + W_EM * EM - LAMBDA * ctx_tokens(q,k) / T_ref
W_EM        = 0.5
LAMBDA_MAIN = 0.10
LAMBDA_GRID = [0.0, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50]

# Policy optimisation (contextual bandit)
POLICY_HIDDEN = [128, 128]
PPO_CFG       = dict(total_samples=200_000, rollout=4096, epochs=10, minibatch=512,
                     lr=3e-4, clip=0.2, ent=0.01, vf=0.5, eval_every=2)
REINFORCE_CFG = dict(total_samples=200_000, batch=512, lr=3e-4, ent=0.01, vf=0.5, eval_every=10)
SUP_CFG       = dict(max_epochs=300, lr=1e-3, batch=256, wd=1e-4, patience=25)

# Baselines
FLARE_K, FLARE_MASK_PROB = 2, 0.4
FLARE_THETAS  = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
SKR_K, SKR_NEIGHBORS = 2, [5, 10, 20]
N_BOOT = 10_000

# Optional reference system
RUN_SELFRAG       = True
SELFRAG_REPO      = "selfrag/selfrag_llama2_7b"
SELFRAG_NDOCS     = 3
SELFRAG_THRESHOLD = 0.2
SELFRAG_W         = dict(rel=1.0, sup=1.0, use=0.5)

FORCE_REPOLICY = False    # True = retrain policies even if result files exist
# ==========================================================================================

In [ ]:
import os, json, gzip, time, math, random, re, string, copy, gc, pickle, platform
from collections import defaultdict, Counter
import numpy as np, pandas as pd
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

for _get in ("kaggle", "colab"):
    try:
        if _get == "kaggle":
            from kaggle_secrets import UserSecretsClient
            os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("llama")
        else:
            from google.colab import userdata
            os.environ["HF_TOKEN"] = userdata.get("llama")
        break
    except Exception:
        pass
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = f"{OUT_ROOT}/data/{DATASET_KEY}"
RUN_DIR  = f"{OUT_ROOT}/runs/{DATASET_KEY}/{MODEL_KEY}"
for d in [DATA_DIR, f"{RUN_DIR}/gen", f"{RUN_DIR}/results", f"{RUN_DIR}/figs", f"{RUN_DIR}/tables"]:
    os.makedirs(d, exist_ok=True)
print("DATA_DIR:", DATA_DIR, "\nRUN_DIR :", RUN_DIR, "\nDEVICE  :", DEVICE)

# ---------------- io helpers ----------------
def save_json(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o))
    os.replace(tmp, path)

def load_json(path, default=None):
    if not os.path.exists(path): return default
    with open(path) as f: return json.load(f)

def read_jsonl(path):
    if not os.path.exists(path): return []
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try: rows.append(json.loads(line))
                except json.JSONDecodeError: pass   # tolerate a torn last line after a crash
    return rows

def append_jsonl(path, rows):
    with open(path, "a") as f:
        for r in rows: f.write(json.dumps(r) + "\n")
        f.flush(); os.fsync(f.fileno())

def save_pkl(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f: pickle.dump(obj, f)
    os.replace(tmp, path)

def load_pkl(path):
    with open(path, "rb") as f: return pickle.load(f)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

STATS_PATH = f"{RUN_DIR}/stage_stats.json"
def log_stage(name, seconds, **extra):
    st = load_json(STATS_PATH, {})
    e = st.get(name, {"wall_s": 0.0})
    e["wall_s"] = e.get("wall_s", 0.0) + seconds
    e.update(extra); st[name] = e; save_json(st, STATS_PATH)

# ---------------- SQuAD-style metrics (max over gold answers) ----------------
_PUNCT = set(string.punctuation)
def normalize_answer(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in _PUNCT)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def _f1(pred, gold):
    p, g = normalize_answer(pred).split(), normalize_answer(gold).split()
    if not p or not g: return float(p == g)
    common = Counter(p) & Counter(g); ns = sum(common.values())
    if ns == 0: return 0.0
    pr, rc = ns / len(p), ns / len(g)
    return 2 * pr * rc / (pr + rc)

def f1_score(pred, golds): return max(_f1(pred, g) for g in golds)
def em_score(pred, golds): return float(max(normalize_answer(pred) == normalize_answer(g) for g in golds))
def acc_score(pred, golds):
    p = normalize_answer(pred)
    return float(any(normalize_answer(g) and normalize_answer(g) in p for g in golds))

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()


os.environ["HF_TOKEN"] = user_secrets.get_secret("llama")
os.environ["HF_USERNAME"] = "Fayaz789"

In [ ]:
# ---------------- model registry and revision pinning ----------------
from huggingface_hub import HfApi, hf_hub_download
MODELS = {
    "llama1b":   dict(repo="meta-llama/Llama-3.2-1B-Instruct",   quant=None),
    "llama3b":   dict(repo="meta-llama/Llama-3.2-3B-Instruct",   quant=None),
    "qwen3b":    dict(repo="Qwen/Qwen2.5-3B-Instruct",           quant=None),
    "llama8b":   dict(repo="meta-llama/Llama-3.1-8B-Instruct",   quant="nf4"),
    "mistral7b": dict(repo="mistralai/Mistral-7B-Instruct-v0.3", quant="nf4"),
}
EMBED_REPO = "BAAI/bge-small-en-v1.5"
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
PINS_PATH = f"{OUT_ROOT}/pins.json"

def pin(kind, repo):
    """Resolve a repo to a commit sha once and reuse it forever (reproducibility)."""
    pins = load_json(PINS_PATH, {})
    key = f"{kind}:{repo}"
    if key not in pins:
        api = HfApi()
        info = api.model_info(repo) if kind == "model" else api.dataset_info(repo)
        pins[key] = info.sha; save_json(pins, PINS_PATH)
    return pins[key]

In [ ]:
def _word_spans(text): return [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]

def chunk_with_spans(text, n):
    spans, out = _word_spans(text), []
    for i in range(0, len(spans), n):
        s, e = spans[i][0], spans[min(i + n, len(spans)) - 1][1]
        out.append((s, e, text[s:e]))
    return out

def grouped_sample(items, sizes, fracs, seed):
    """Assign whole groups to splits (by fraction of groups), then sample n questions per split."""
    rng = np.random.RandomState(seed)
    groups = sorted({it["group"] for it in items}); rng.shuffle(groups)
    by_g = defaultdict(list)
    for it in items: by_g[it["group"]].append(it)
    n_g = len(groups); cut1 = int(round(fracs["test"] * n_g)); cut2 = cut1 + int(round(fracs["val"] * n_g))
    gsplit = {"test": groups[:cut1], "val": groups[cut1:cut2], "train": groups[cut2:]}
    out = {}
    for s in ["train", "val", "test"]:
        pool = [it for g in gsplit[s] for it in by_g[g]]
        pool.sort(key=lambda x: x["qid"])
        if len(pool) < sizes[s]:
            print(f"[warn] {s}: pool {len(pool)} < requested {sizes[s]}; using all")
            out[s] = pool
        else:
            sel = rng.choice(len(pool), sizes[s], replace=False)
            out[s] = [pool[i] for i in sorted(sel)]
    tr = {it["group"] for it in out["train"]}; va = {it["group"] for it in out["val"]}; te = {it["group"] for it in out["test"]}
    assert not (tr & va) and not (tr & te) and not (va & te), "group leakage across splits"
    return out

def build_squad():
    from datasets import load_dataset
    rev = pin("dataset", "rajpurkar/squad")
    df = load_dataset("rajpurkar/squad", split="train", revision=rev).to_pandas()
    passages, ctx_chunks = [], {}
    for (title, ctx), _ in df.groupby(["title", "context"], sort=True):
        chunks = []
        for s, e, txt in chunk_with_spans(ctx, PASSAGE_WORDS):
            pid = f"sq{len(passages)}"
            passages.append(dict(pid=pid, title=title.replace("_", " "), text=txt))
            chunks.append((s, e, pid))
        ctx_chunks[(title, ctx)] = chunks
    items = []
    for r in df.itertuples(index=False):
        texts = [str(t) for t in r.answers["text"]]; starts = [int(x) for x in r.answers["answer_start"]]
        if not texts: continue
        gold = set()
        for t, st in zip(texts, starts):
            en = st + len(t)
            for s, e, pid in ctx_chunks[(r.title, r.context)]:
                if s < en and st < e: gold.add(pid)
        items.append(dict(qid=str(r.id), question=r.question.strip(), answers=texts,
                          gold_pids=sorted(gold), group=r.title))
    return items, passages, {"squad_revision": rev}

def build_nq():
    rev = pin("dataset", "Tevatron/wikipedia-nq")
    path = hf_hub_download("Tevatron/wikipedia-nq", "nq-dev.jsonl.gz", repo_type="dataset", revision=rev)
    corpus, items = {}, []
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in tqdm(f, desc="Parsing NQ"):
            r = json.loads(line)
            pos = r.get("positive_passages") or []
            neg = (r.get("negative_passages") or [])[:NQ_NEGS_PER_Q]
            for p in pos + neg:
                pid = f"nq{p['docid']}"
                if pid not in corpus:
                    corpus[pid] = dict(pid=pid, title=p.get("title", ""), text=p["text"])
            ans = [a for a in (r.get("answers") or []) if str(a).strip()]
            if not pos or not ans: continue
            items.append(dict(qid=f"nq{r['query_id']}", question=r["query"].strip(), answers=ans,
                              gold_pids=[f"nq{p['docid']}" for p in pos],
                              group=pos[0].get("title") or f"nq{pos[0]['docid']}"))
    passages = sorted(corpus.values(), key=lambda p: p["pid"])
    return items, passages, {"nq_revision": rev, "nq_file": "nq-dev.jsonl.gz"}

SPLITS_PATH, CORPUS_PATH = f"{DATA_DIR}/splits.json", f"{DATA_DIR}/corpus.jsonl"
if not (os.path.exists(SPLITS_PATH) and os.path.exists(CORPUS_PATH)):
    t0 = time.time()
    items, passages, meta = build_squad() if DATASET_KEY == "squad" else build_nq()
    splits = grouped_sample(items, dict(train=N_TRAIN, val=N_VAL, test=N_TEST), GROUP_FRACS, DATA_SEED)
    if os.path.exists(CORPUS_PATH): os.remove(CORPUS_PATH)
    append_jsonl(CORPUS_PATH, passages)
    save_json(splits, SPLITS_PATH)
    meta.update(n_pool=len(items), n_passages=len(passages), data_seed=DATA_SEED,
                sizes={s: len(v) for s, v in splits.items()}, passage_words=PASSAGE_WORDS)
    save_json(meta, f"{DATA_DIR}/data_manifest.json")
    print(f"Built in {time.time()-t0:.0f}s")

SPLITS   = load_json(SPLITS_PATH)
PASSAGES = read_jsonl(CORPUS_PATH)
PID2IDX  = {p["pid"]: i for i, p in enumerate(PASSAGES)}
print({s: len(v) for s, v in SPLITS.items()}, "| corpus:", len(PASSAGES))
print(json.dumps(load_json(f"{DATA_DIR}/data_manifest.json"), indent=1))

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

class Retriever:
    def __init__(self):
        self.enc = SentenceTransformer(EMBED_REPO, revision=pin("model", EMBED_REPO), device=DEVICE)
        emb_path = f"{DATA_DIR}/corpus_emb.npy"
        if os.path.exists(emb_path):
            emb = np.load(emb_path).astype(np.float32)
        else:
            texts = [f"{p['title']}\n{p['text']}" for p in PASSAGES]
            emb = self.enc.encode(texts, batch_size=256, normalize_embeddings=True,
                                  convert_to_numpy=True, show_progress_bar=True).astype(np.float32)
            np.save(emb_path, emb)
        self.dim = emb.shape[1]
        self.index = faiss.IndexFlatIP(self.dim); self.index.add(emb)

    def encode_queries(self, qs, bs=256):
        return self.enc.encode([BGE_QUERY_PREFIX + q for q in qs], batch_size=bs, normalize_embeddings=True,
                               convert_to_numpy=True, show_progress_bar=False).astype(np.float32)

    def search(self, qemb, n):
        return self.index.search(qemb, n)

_RETRIEVER = None
def get_retriever():
    global _RETRIEVER
    if _RETRIEVER is None: _RETRIEVER = Retriever()
    return _RETRIEVER

RET = {}
for split, items in SPLITS.items():
    path = f"{DATA_DIR}/retrieval_{split}.npz"
    if not os.path.exists(path):
        R = get_retriever()
        qemb = R.encode_queries([it["question"] for it in items])
        sc, ix = R.search(qemb, TOPN_RETRIEVE)
        np.savez(path, qemb=qemb, scores=sc, idx=ix)
    z = np.load(path); RET[split] = dict(qemb=z["qemb"], scores=z["scores"], idx=z["idx"])

def retrieval_metrics(split):
    items, ix = SPLITS[split], RET[split]["idx"]
    rec = {k: [] for k in [1, 2, 3, 5, 10]}; mrr, ndcg, hit = [], [], {k: [] for k in [1, 2, 3]}
    for j, it in enumerate(items):
        gold = {PID2IDX[p] for p in it["gold_pids"] if p in PID2IDX}
        rel = [int(i in gold) for i in ix[j]]
        for k in rec: rec[k].append(float(any(rel[:k])))
        rr = next((1.0 / (r + 1) for r, v in enumerate(rel) if v), 0.0); mrr.append(rr)
        dcg = sum(v / math.log2(r + 2) for r, v in enumerate(rel))
        idcg = sum(1 / math.log2(r + 2) for r in range(min(len(gold), len(rel))))
        ndcg.append(dcg / idcg if idcg > 0 else 0.0)
        golds = [normalize_answer(a) for a in it["answers"]]
        for k in hit:
            txt = normalize_answer(" ".join(PASSAGES[i]["text"] for i in ix[j, :k]))
            hit[k].append(float(any(g and g in txt for g in golds)))
    out = {f"Recall@{k}": np.mean(v) for k, v in rec.items()}
    out.update({"MRR@10": np.mean(mrr), "nDCG@10": np.mean(ndcg)})
    out.update({f"AnswerHit@{k}": np.mean(v) for k, v in hit.items()})
    return out

RETR_METRICS = {s: retrieval_metrics(s) for s in SPLITS}
pd.DataFrame(RETR_METRICS).T.round(3)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

SYSTEM_PROMPT = ("You are a question answering assistant. Answer with the shortest possible span "
                 "(at most 5 words). Output only the answer, with no explanation.")

def build_messages(question, passages):
    if passages:
        docs = "\n\n".join(f"[{i+1}] {p['title']}: {p['text']}" for i, p in enumerate(passages))
        user = f"Use the documents if they are helpful.\n\nDocuments:\n{docs}\n\nQuestion: {question}"
    else:
        user = f"Question: {question}"
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]

def clean_answer(t):
    t = t.strip().split("\n")[0].strip()
    t = re.sub(r"^(final answer|answer)\s*[:\-]\s*", "", t, flags=re.I)
    t = t.strip(" \"'`*")
    return re.sub(r"[.\u3002]+$", "", t).strip()

def render_prompt(tok, question, passages=None):
    msgs = build_messages(question, passages)
    try:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, date_string="26 Jul 2024")
    except Exception:   # templates without a system role
        merged = [{"role": "user", "content": msgs[0]["content"] + "\n\n" + msgs[1]["content"]}]
        return tok.apply_chat_template(merged, tokenize=False, add_generation_prompt=True)

class LLM:
    def __init__(self, key):
        spec = MODELS[key]; self.repo = spec["repo"]; self.sha = pin("model", self.repo)
        self.tok = AutoTokenizer.from_pretrained(self.repo, revision=self.sha, padding_side="left")
        if self.tok.pad_token is None: self.tok.pad_token = self.tok.eos_token
        kw = dict(revision=self.sha, device_map={"": 0}, torch_dtype=torch.float16)
        if spec["quant"] == "nf4":
            kw["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
        self.model = AutoModelForCausalLM.from_pretrained(self.repo, **kw).eval()
        eos = self.model.generation_config.eos_token_id
        eos = eos if isinstance(eos, list) else [eos]
        self.eos_ids = sorted({int(e) for e in eos if e is not None} | {int(self.tok.eos_token_id)})
        self.pad_id = int(self.tok.pad_token_id)
        self.stop = set(self.eos_ids) | {self.pad_id}
        self.quant = spec["quant"]

    def prompt(self, question, passages=None):
        return render_prompt(self.tok, question, passages)

    @torch.inference_mode()
    def _gen_batch(self, prompts, with_scores):
        enc = self.tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(self.model.device)
        out = self.model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, num_beams=1,
                                  temperature=None, top_p=None, top_k=None,
                                  pad_token_id=self.pad_id, eos_token_id=self.eos_ids,
                                  return_dict_in_generate=True, output_scores=with_scores)
        L = enc["input_ids"].shape[1]; gen = out.sequences[:, L:]
        n_in = enc["attention_mask"].sum(1).tolist(); B, T = gen.shape
        if with_scores:
            chosen = torch.zeros(B, T, device=gen.device)
            for t, s in enumerate(out.scores):
                lp = torch.log_softmax(s.float(), -1)
                chosen[:, t] = lp.gather(1, gen[:, t:t+1]).squeeze(1)
                if t == 0:
                    p = lp.exp(); top2 = p.topk(2, -1).values
                    f_maxp = top2[:, 0].tolist(); f_margin = (top2[:, 0] - top2[:, 1]).tolist()
                    f_ent = (-(p * lp.clamp(min=-1e4)).sum(-1)).tolist()
            chosen = chosen.tolist()
        res, gl = [], gen.tolist()
        for b in range(B):
            ids = gl[b]; n = next((t for t, i in enumerate(ids) if i in self.stop), len(ids))
            text = self.tok.decode(ids[:n], skip_special_tokens=True)
            r = dict(raw=text, answer=clean_answer(text), n_in=int(n_in[b]), n_out=int(n + (1 if n < len(ids) else 0)))
            if with_scores:
                lps = chosen[b][:n] if T > 0 else []
                r.update(tok_logp=[round(x, 4) for x in lps],
                         tok_str=[self.tok.decode([i]) for i in ids[:n]],
                         first_maxp=f_maxp[b] if T > 0 else 0.0, first_ent=f_ent[b] if T > 0 else 0.0,
                         first_margin=f_margin[b] if T > 0 else 0.0,
                         mean_logp=float(np.mean(lps)) if lps else 0.0, min_logp=float(np.min(lps)) if lps else 0.0)
            res.append(r)
        return res

    def generate_many(self, prompts, bs, with_scores=False, progress=True):
        order = np.argsort([len(p) for p in prompts])[::-1]
        out, i, cur = [None] * len(prompts), 0, bs
        pbar = tqdm(total=len(prompts), disable=not progress, leave=False)
        while i < len(order):
            idx = order[i:i + cur]
            try:
                res = self._gen_batch([prompts[j] for j in idx], with_scores)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if cur == 1: raise
                cur = max(1, cur // 2); print(f"[OOM] batch size -> {cur}"); continue
            for j, r in zip(idx, res): out[j] = r
            i += len(idx); pbar.update(len(idx))
        pbar.close(); return out

_LLM = None
def get_llm():
    global _LLM
    if _LLM is None:
        t0 = time.time(); _LLM = LLM(MODEL_KEY); print(f"Loaded {MODEL_KEY} @ {_LLM.sha[:10]} in {time.time()-t0:.0f}s")
    return _LLM

def ctx_passages(split, j, k):
    return [PASSAGES[i] for i in RET[split]["idx"][j, :k]] if k > 0 else None

In [ ]:
def gen_path(split, k): return f"{RUN_DIR}/gen/{split}_{'flare' if k == 'flare' else f'k{k}'}.jsonl"

def done_ids(path): return {r["qid"] for r in read_jsonl(path)}

def flare_query(question, draft):
    kept = [s for s, lp in zip(draft["tok_str"], draft["tok_logp"]) if math.exp(lp) >= FLARE_MASK_PROB]
    return (question + " " + "".join(kept).strip()).strip()

def run_generation(split, k):
    items, path = SPLITS[split], gen_path(split, k); done = done_ids(path)
    todo = [j for j, it in enumerate(items) if it["qid"] not in done]
    if not todo: return 0
    llm = get_llm()
    for c in tqdm(range(0, len(todo), GEN_CHUNK), desc=f"{split} k={k}"):
        js = todo[c:c + GEN_CHUNK]
        prompts = [llm.prompt(items[j]["question"], ctx_passages(split, j, k)) for j in js]
        outs = llm.generate_many(prompts, GEN_BATCH[k], with_scores=(k == 0), progress=False)
        append_jsonl(path, [dict(qid=items[j]["qid"], k=k, **o) for j, o in zip(js, outs)])
    return len(todo)

def run_flare(split):
    items, path = SPLITS[split], gen_path(split, "flare"); done = done_ids(path)
    todo = [j for j, it in enumerate(items) if it["qid"] not in done]
    if not todo: return 0
    drafts = {r["qid"]: r for r in read_jsonl(gen_path(split, 0))}
    llm, R = get_llm(), get_retriever()
    for c in tqdm(range(0, len(todo), GEN_CHUNK), desc=f"{split} flare"):
        js = todo[c:c + GEN_CHUNK]
        fq = [flare_query(items[j]["question"], drafts[items[j]["qid"]]) for j in js]
        _, fix = R.search(R.encode_queries(fq), FLARE_K)
        prompts = [llm.prompt(items[j]["question"], [PASSAGES[i] for i in fix[n]]) for n, j in enumerate(js)]
        outs = llm.generate_many(prompts, GEN_BATCH[FLARE_K], with_scores=False, progress=False)
        append_jsonl(path, [dict(qid=items[j]["qid"], flare_query=fq[n], flare_pids=[PASSAGES[i]["pid"] for i in fix[n]], **o)
                            for n, (j, o) in enumerate(zip(js, outs))])
    return len(todo)

t0 = time.time(); n_new = 0
if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
for split in ["test", "val", "train"]:          # test first: sanity-check early
    for k in K_OPTIONS: n_new += run_generation(split, k)
    n_new += run_flare(split)
if n_new:
    log_stage("precompute_generation", time.time() - t0, new_generations=n_new,
              peak_gpu_mb=(torch.cuda.max_memory_allocated() / 2**20) if torch.cuda.is_available() else None)
print("Generation cache complete.", json.dumps(load_json(STATS_PATH, {}), indent=1))

In [ ]:
def quick_static(split):
    items = SPLITS[split]; rows = {}
    for k in K_OPTIONS:
        g = {r["qid"]: r for r in read_jsonl(gen_path(split, k))}
        rows[f"k={k}"] = dict(F1=np.mean([f1_score(g[it["qid"]]["answer"], it["answers"]) for it in items]),
                              EM=np.mean([em_score(g[it["qid"]]["answer"], it["answers"]) for it in items]),
                              prompt_tokens=np.mean([g[it["qid"]]["n_in"] for it in items]))
    return pd.DataFrame(rows).T.round(3)

display(quick_static("val"))
g = {k: {r["qid"]: r for r in read_jsonl(gen_path("test", k))} for k in K_OPTIONS}
for it in SPLITS["test"][:6]:
    print(f"\nQ: {it['question']}\n   gold: {it['answers'][:3]}")
    for k in K_OPTIONS: print(f"   k={k}: {g[k][it['qid']]['raw'][:90]!r}")
_tok = _LLM.tok if _LLM is not None else AutoTokenizer.from_pretrained(MODELS[MODEL_KEY]["repo"], revision=pin("model", MODELS[MODEL_KEY]["repo"]))
ex = render_prompt(_tok, SPLITS["test"][0]["question"], ctx_passages("test", 0, 2))
open(f"{RUN_DIR}/prompt_example_k2.txt", "w").write(ex)
print("\nPrompt example saved (for the paper's prompt figure):\n", ex[:1500])

In [ ]:
def timed(fn):
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t = time.perf_counter(); r = fn()
    if torch.cuda.is_available(): torch.cuda.synchronize()
    return r, (time.perf_counter() - t) * 1000.0

LAT_PATH = f"{RUN_DIR}/latency_test.jsonl"
def profile_latency():
    items = SPLITS["test"][:LATENCY_N]; done = done_ids(LAT_PATH)
    todo = [j for j, it in enumerate(items) if it["qid"] not in done]
    if not todo: return
    llm, R = get_llm(), get_retriever()
    drafts = {r["qid"]: r for r in read_jsonl(gen_path("test", 0))}
    for _ in range(3):  # warm-up
        llm.generate_many([llm.prompt("What is the capital of France?")], 1, progress=False)
        R.search(R.encode_queries(["warm up"]), TOPN_RETRIEVE)
    buf = []
    for j in tqdm(todo, desc="latency"):
        it = items[j]; q = it["question"]; row = dict(qid=it["qid"])
        qe, row["t_embed"] = timed(lambda: R.encode_queries([q], bs=1))
        (_, ix), row["t_search"] = timed(lambda: R.search(qe, TOPN_RETRIEVE))
        for k in K_OPTIONS:
            ctx = [PASSAGES[i] for i in ix[0, :k]] if k > 0 else None
            _, row[f"t_gen_{k}"] = timed(lambda: llm.generate_many([llm.prompt(q, ctx)], 1, with_scores=(k == 0), progress=False))
        fq = flare_query(q, drafts[it["qid"]])
        fqe, row["t_flare_embed"] = timed(lambda: R.encode_queries([fq], bs=1))
        (_, fix), row["t_flare_search"] = timed(lambda: R.search(fqe, FLARE_K))
        fctx = [PASSAGES[i] for i in fix[0]]
        _, row["t_flare_gen"] = timed(lambda: llm.generate_many([llm.prompt(q, fctx)], 1, progress=False))
        buf.append(row)
        if len(buf) >= 10: append_jsonl(LAT_PATH, buf); buf = []
    if buf: append_jsonl(LAT_PATH, buf)

t0 = time.time(); profile_latency(); log_stage("latency_profiling", time.time() - t0)
lat_df = pd.DataFrame(read_jsonl(LAT_PATH))
lat_df.drop(columns="qid").describe().loc[["mean", "50%", "std"]].round(1)

In [ ]:
if _LLM is not None:
    del _LLM.model; _LLM = None
gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("LLM released.")

In [ ]:
WH = ["who", "what", "when", "where", "which", "how", "why"]
UNC_NAMES   = ["first_maxp", "first_ent", "first_margin", "mean_logp", "min_logp", "draft_len"]
RET_NAMES   = ["s1", "s1_minus_s2", "mean_s1_3", "s1_minus_s10"]
QUERY_NAMES = ["n_words", "cap_ratio", "has_digit"] + [f"wh_{w}" for w in WH] + ["wh_other"]
FEATURE_NAMES  = UNC_NAMES + RET_NAMES + QUERY_NAMES
FEATURE_GROUPS = {"unc": UNC_NAMES, "ret": RET_NAMES, "query": QUERY_NAMES}

def query_type(q):
    toks = q.lower().split()
    return next((w for w in toks[:3] if w in WH), next((w for w in toks if w in WH), "other"))

def query_feats(q):
    words = q.split(); n = len(words); wt = query_type(q)
    return [n, sum(w[:1].isupper() for w in words[1:]) / max(n - 1, 1), float(any(c.isdigit() for c in q))] + \
           [float(wt == w) for w in WH] + [float(wt == "other")]

def build_tables(split):
    path = f"{RUN_DIR}/tables_{split}.pkl"
    if os.path.exists(path): return load_pkl(path)
    items = SPLITS[split]; n, K = len(items), len(K_OPTIONS)
    T = {m: np.zeros((n, K)) for m in ["F1", "EM", "ACC", "NIN", "NOUT"]}
    ANS = [[None] * K for _ in range(n)]
    for ki, k in enumerate(K_OPTIONS):
        g = {r["qid"]: r for r in read_jsonl(gen_path(split, k))}
        for j, it in enumerate(items):
            r = g[it["qid"]]; a = r["answer"]; ANS[j][ki] = a
            T["F1"][j, ki] = f1_score(a, it["answers"]); T["EM"][j, ki] = em_score(a, it["answers"])
            T["ACC"][j, ki] = acc_score(a, it["answers"]); T["NIN"][j, ki] = r["n_in"]; T["NOUT"][j, ki] = r["n_out"]
    gf = {r["qid"]: r for r in read_jsonl(gen_path(split, "flare"))}
    g0 = {r["qid"]: r for r in read_jsonl(gen_path(split, 0))}
    FL = {m: np.zeros(n) for m in ["F1", "EM", "ACC", "NIN", "NOUT", "MINP"]}; FANS = []
    X = np.zeros((n, len(FEATURE_NAMES)), dtype=np.float32); sc = RET[split]["scores"]
    for j, it in enumerate(items):
        r = gf[it["qid"]]; a = r["answer"]; FANS.append(a)
        FL["F1"][j] = f1_score(a, it["answers"]); FL["EM"][j] = em_score(a, it["answers"])
        FL["ACC"][j] = acc_score(a, it["answers"]); FL["NIN"][j] = r["n_in"]; FL["NOUT"][j] = r["n_out"]
        d = g0[it["qid"]]; FL["MINP"][j] = math.exp(d["min_logp"]) if d["tok_logp"] else 0.0
        unc = [d["first_maxp"], d["first_ent"], d["first_margin"], d["mean_logp"], d["min_logp"], len(d["tok_logp"])]
        ret = [sc[j, 0], sc[j, 0] - sc[j, 1], sc[j, :3].mean(), sc[j, 0] - sc[j, -1]]
        X[j] = np.nan_to_num(np.array(unc + ret + query_feats(it["question"]), dtype=np.float32))
    tab = dict(split=split, qids=[it["qid"] for it in items], questions=[it["question"] for it in items],
               golds=[it["answers"] for it in items], qtype=[query_type(it["question"]) for it in items],
               X=X, ANS=ANS, FANS=FANS, FL=FL, **T)
    save_pkl(tab, path); return tab

TAB = {s: build_tables(s) for s in ["train", "val", "test"]}
MU, SD = TAB["train"]["X"].mean(0), TAB["train"]["X"].std(0) + 1e-6
def feats(tab, groups=("unc", "ret", "query"), mu=None, sd=None):
    mu = MU if mu is None else mu; sd = SD if sd is None else sd
    cols = [FEATURE_NAMES.index(f) for g in groups for f in FEATURE_GROUPS[g]]
    return ((tab["X"] - mu) / sd)[:, cols].astype(np.float32)

T_REF = float(np.mean(TAB["train"]["NIN"][:, -1] - TAB["train"]["NIN"][:, 0]))   # mean ctx tokens at k_max
print("T_ref (mean context tokens at k_max, train):", round(T_REF, 1))

# latency arrays aligned to the test split (NaN where not profiled)
LAT = {}
_lat = {r["qid"]: r for r in read_jsonl(LAT_PATH)}
for key in ["t_embed", "t_search", "t_flare_embed", "t_flare_search", "t_flare_gen"] + [f"t_gen_{k}" for k in K_OPTIONS]:
    LAT[key] = np.array([_lat.get(q, {}).get(key, np.nan) for q in TAB["test"]["qids"]])
print("profiled test queries:", int(np.isfinite(LAT["t_gen_0"]).sum()))

In [ ]:
from torch.distributions import Categorical
from scipy import stats as sps

def reward_matrix(tab, lam=LAMBDA_MAIN, w_em=W_EM, kind="clean", tref=None):
    ctx = (tab["NIN"] - tab["NIN"][:, :1]) / (T_REF if tref is None else tref)
    if kind == "clean":
        return tab["F1"] + w_em * tab["EM"] - lam * ctx
    if kind == "shaped":   # original manuscript reward (action-history term removed: it made queries dependent)
        F1 = tab["F1"]; imp = F1 - F1[:, :1]
        conf = tab["X"][:, FEATURE_NAMES.index("first_maxp")]; sim = tab["X"][:, FEATURE_NAMES.index("s1")]
        R = np.zeros_like(F1)
        for ki, k in enumerate(K_OPTIONS):
            r = 5.0 * imp[:, ki] - 2.0 * 0.1 * k; im = imp[:, ki]
            if k == 0:   r += np.where(conf > 0.8, 0.3, np.where(conf < 0.4, -0.4, 0.0))
            elif k == 1: r += np.where(sim > 0.8, 0.4, np.where(im > 0.2, 0.3, np.where((sim < 0.3) & (im < 0.1), -0.2, 0.0)))
            elif k == 2: r += np.where((conf >= .4) & (conf <= .7) & (sim >= .4) & (sim <= .7), 0.2,
                                np.where(im > 0.3, 0.4, np.where((im < 0.05) & (conf > 0.8), -0.3, 0.0)))
            elif k == 3: r += np.where((conf < 0.3) & (sim < 0.3), np.where(im > 0.4, 0.5, np.where(im > 0.2, 0.2, 0.0)), -0.5)
            eff = im / (k + 1e-6); r += np.where(eff > 0.1, 2.0 * eff, 0.0)
            R[:, ki] = np.clip(r, -5, 5)
        return R
    raise ValueError(kind)

def evaluate(tab, ki, draft=False, upfront=False, policy_ms=0.0, kind="std", trig=None, knn_ms=0.0, lam=LAMBDA_MAIN):
    n = len(ki); ar = np.arange(n); kv = np.array(K_OPTIONS)[ki]
    gtok = lambda c: tab["NIN"][:, c] + tab["NOUT"][:, c]
    if kind == "flare":
        FL = tab["FL"]; t = trig.astype(bool)
        f1 = np.where(t, FL["F1"], tab["F1"][:, 0]); em = np.where(t, FL["EM"], tab["EM"][:, 0]); acc = np.where(t, FL["ACC"], tab["ACC"][:, 0])
        total = gtok(0) + t * (FL["NIN"] + FL["NOUT"]); final = np.where(t, FL["NIN"], tab["NIN"][:, 0])
        ctx = np.where(t, FL["NIN"] - tab["NIN"][:, 0], 0); kv = np.where(t, FLARE_K, 0)
    else:
        f1, em, acc = tab["F1"][ar, ki], tab["EM"][ar, ki], tab["ACC"][ar, ki]
        need_gen = (kv > 0) | (not draft)
        total = gtok(0) * draft + need_gen * (tab["NIN"][ar, ki] + tab["NOUT"][ar, ki])
        final = tab["NIN"][ar, ki]; ctx = tab["NIN"][ar, ki] - tab["NIN"][:, 0]
    out = dict(f1=f1, em=em, acc=acc, k=kv.astype(float), total_tokens=total.astype(float),
               final_prompt_tokens=final.astype(float), ctx_tokens=ctx.astype(float))
    out["reward"] = f1 + W_EM * em - lam * ctx / T_REF
    if tab["split"] == "test":
        L = LAT; g = lambda k: L[f"t_gen_{k}"]
        if kind == "flare":
            lat = g(0) + t * (L["t_flare_embed"] + L["t_flare_search"] + L["t_flare_gen"])
        elif kind == "skr":
            lat = L["t_embed"] + knn_ms + (kv > 0) * L["t_search"] + np.choose(ki, [g(k) for k in K_OPTIONS])
        else:
            gk = np.choose(ki, [g(k) for k in K_OPTIONS])
            lat = draft * g(0) + (upfront | (kv > 0)) * (L["t_embed"] + L["t_search"]) + policy_ms + need_gen * gk
        out["latency_ms"] = lat.astype(float)
    return out

def summarize(ev):
    s = {m: float(np.nanmean(ev[m])) for m in ["f1", "em", "acc", "k", "total_tokens", "final_prompt_tokens", "ctx_tokens", "reward"]}
    if "latency_ms" in ev: s["latency_ms"] = float(np.nanmean(ev["latency_ms"]))
    s["k_dist"] = {int(k): int(c) for k, c in zip(*np.unique(ev["k"], return_counts=True))}
    return s

# ---------------- neural policies ----------------
class MLP(nn.Module):
    def __init__(self, d, out, hidden=POLICY_HIDDEN):
        super().__init__(); layers, h = [], d
        for w in hidden: layers += [nn.Linear(h, w), nn.ReLU()]; h = w
        layers.append(nn.Linear(h, out)); self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def greedy(actor, X):
    with torch.no_grad(): return actor(torch.as_tensor(X)).argmax(-1).numpy()

def policy_latency_ms(actor, d):
    x = torch.zeros(1, d); [actor(x) for _ in range(20)]
    ts = []
    for _ in range(200):
        t = time.perf_counter()
        with torch.no_grad(): actor(x)
        ts.append((time.perf_counter() - t) * 1000)
    return float(np.median(ts))

def _val_reward(actor, Xv, Rv): return float(Rv[np.arange(len(Rv)), greedy(actor, Xv)].mean())

def train_ppo(Xtr, Rtr, Xva, Rva, seed, cfg=PPO_CFG):
    set_seed(seed); g = torch.Generator().manual_seed(seed)
    X, R = torch.tensor(Xtr), torch.tensor(Rtr, dtype=torch.float32)
    actor, critic = MLP(X.shape[1], R.shape[1]), MLP(X.shape[1], 1)
    params = list(actor.parameters()) + list(critic.parameters()); opt = torch.optim.Adam(params, lr=cfg["lr"])
    n_upd = max(1, cfg["total_samples"] // cfg["rollout"]); best, best_state, log = -1e9, None, []
    for u in range(n_upd):
        idx = torch.randint(len(X), (cfg["rollout"],), generator=g); s = X[idx]
        with torch.no_grad():
            probs = torch.softmax(actor(s), -1); a = torch.multinomial(probs, 1, generator=g).squeeze(1)
            logp_old = torch.log(probs.gather(1, a[:, None]).squeeze(1) + 1e-12)
            r = R[idx, a]; adv = r - critic(s).squeeze(1); adv = (adv - adv.mean()) / (adv.std() + 1e-8)
        for _ in range(cfg["epochs"]):
            for mb in torch.randperm(len(idx), generator=g).split(cfg["minibatch"]):
                lp = torch.log_softmax(actor(s[mb]), -1); nlp = lp.gather(1, a[mb, None]).squeeze(1)
                ratio = torch.exp(nlp - logp_old[mb])
                pg = -torch.min(ratio * adv[mb], torch.clamp(ratio, 1 - cfg["clip"], 1 + cfg["clip"]) * adv[mb]).mean()
                ent = -(lp.exp() * lp).sum(-1).mean()
                vl = ((critic(s[mb]).squeeze(1) - r[mb]) ** 2).mean()
                loss = pg + cfg["vf"] * vl - cfg["ent"] * ent
                opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(params, 0.5); opt.step()
        if (u + 1) % cfg["eval_every"] == 0 or u == n_upd - 1:
            v = _val_reward(actor, Xva, Rva)
            log.append(dict(samples=(u + 1) * cfg["rollout"], val_reward=v, train_reward=float(r.mean())))
            if v > best: best, best_state = v, copy.deepcopy(actor.state_dict())
    actor.load_state_dict(best_state); return actor, log

def train_reinforce(Xtr, Rtr, Xva, Rva, seed, cfg=REINFORCE_CFG):
    set_seed(seed); g = torch.Generator().manual_seed(seed)
    X, R = torch.tensor(Xtr), torch.tensor(Rtr, dtype=torch.float32)
    actor, critic = MLP(X.shape[1], R.shape[1]), MLP(X.shape[1], 1)
    params = list(actor.parameters()) + list(critic.parameters()); opt = torch.optim.Adam(params, lr=cfg["lr"])
    n_upd = max(1, cfg["total_samples"] // cfg["batch"]); best, best_state, log = -1e9, None, []
    for u in range(n_upd):
        idx = torch.randint(len(X), (cfg["batch"],), generator=g); s = X[idx]
        lp = torch.log_softmax(actor(s), -1)
        a = torch.multinomial(lp.exp().detach(), 1, generator=g).squeeze(1)
        r = R[idx, a]; v = critic(s).squeeze(1)
        adv = (r - v).detach()
        loss = -(lp.gather(1, a[:, None]).squeeze(1) * adv).mean() + cfg["vf"] * ((v - r) ** 2).mean() \
               - cfg["ent"] * (-(lp.exp() * lp).sum(-1).mean())
        opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(params, 0.5); opt.step()
        if (u + 1) % cfg["eval_every"] == 0 or u == n_upd - 1:
            v_ = _val_reward(actor, Xva, Rva)
            log.append(dict(samples=(u + 1) * cfg["batch"], val_reward=v_, train_reward=float(r.mean())))
            if v_ > best: best, best_state = v_, copy.deepcopy(actor.state_dict())
    actor.load_state_dict(best_state); return actor, log

def train_supervised(Xtr, Rtr, Xva, Rva, seed, cfg=SUP_CFG):
    """Supervised k-predictor: labels = cost-aware oracle k (argmax reward; ties -> smaller k)."""
    set_seed(seed); g = torch.Generator().manual_seed(seed)
    X, y = torch.tensor(Xtr), torch.tensor(Rtr.argmax(1))
    actor = MLP(X.shape[1], Rtr.shape[1]); opt = torch.optim.Adam(actor.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    best, best_state, wait, log = -1e9, None, 0, []
    for ep in range(cfg["max_epochs"]):
        for mb in torch.randperm(len(X), generator=g).split(cfg["batch"]):
            loss = nn.functional.cross_entropy(actor(X[mb]), y[mb]); opt.zero_grad(); loss.backward(); opt.step()
        v = _val_reward(actor, Xva, Rva); log.append(dict(epoch=ep + 1, val_reward=v))
        if v > best + 1e-6: best, best_state, wait = v, copy.deepcopy(actor.state_dict()), 0
        else:
            wait += 1
            if wait >= cfg["patience"]: break
    actor.load_state_dict(best_state); return actor, log

TRAINERS = {"ppo": train_ppo, "reinforce": train_reinforce, "supervised": train_supervised}

def run_learned(name, algo, groups=("unc", "ret", "query"), actions=None, lam=LAMBDA_MAIN, w_em=W_EM,
                reward_kind="clean", seeds=SEEDS, save=True):
    """Train on train, select checkpoint on val, evaluate on test. Returns {seed: eval}."""
    path = f"{RUN_DIR}/results/{name}.pkl"
    if save and os.path.exists(path) and not FORCE_REPOLICY: return load_pkl(path)
    actions = list(range(len(K_OPTIONS))) if actions is None else [K_OPTIONS.index(k) for k in actions]
    Rtr = reward_matrix(TAB["train"], lam, w_em, reward_kind)[:, actions]
    Rva = reward_matrix(TAB["val"], lam, w_em, reward_kind)[:, actions]
    Xtr, Xva, Xte = feats(TAB["train"], groups), feats(TAB["val"], groups), feats(TAB["test"], groups)
    out = {}
    for s in seeds:
        t0 = time.time(); actor, log = TRAINERS[algo](Xtr, Rtr, Xva, Rva, s)
        train_s = time.time() - t0
        ki = np.array(actions)[greedy(actor, Xte)]
        ev = evaluate(TAB["test"], ki, draft="unc" in groups, upfront="ret" in groups,
                      policy_ms=policy_latency_ms(actor, Xte.shape[1]))
        ev["train_log"], ev["train_seconds"] = log, train_s
        ev["n_params"] = sum(p.numel() for p in actor.parameters()); out[s] = ev
    if save: save_pkl(out, path)
    return out

# ---------------- non-learned baselines ----------------
def run_static(k):
    ki = np.full(len(TAB["test"]["qids"]), K_OPTIONS.index(k)); return {0: evaluate(TAB["test"], ki)}

def run_static_best(select="reward", lam=LAMBDA_MAIN):
    Rva = reward_matrix(TAB["val"], lam) if select == "reward" else TAB["val"]["F1"]
    kb = int(Rva.mean(0).argmax()); ev = run_static(K_OPTIONS[kb]); ev[0]["selected_k"] = K_OPTIONS[kb]; return ev

def run_oracle(lam=LAMBDA_MAIN):
    ki = reward_matrix(TAB["test"], lam).argmax(1); return {0: evaluate(TAB["test"], ki)}

def run_random(seeds=SEEDS):
    out = {}
    for s in seeds:
        rng = np.random.RandomState(s); out[s] = evaluate(TAB["test"], rng.randint(len(K_OPTIONS), size=len(TAB["test"]["qids"])))
    return out

def run_threshold(lam=LAMBDA_MAIN):
    """Uncertainty-threshold rule: answer with the draft if first-token confidence >= tau, else retrieve k."""
    c = lambda t: t["X"][:, FEATURE_NAMES.index("first_maxp")]
    Rva = reward_matrix(TAB["val"], lam); best = (-1e9, None, None)
    for tau in np.unique(np.quantile(c(TAB["val"]), np.linspace(0, 1, 41))):
        for kk in range(1, len(K_OPTIONS)):
            ki = np.where(c(TAB["val"]) >= tau, 0, kk); v = Rva[np.arange(len(ki)), ki].mean()
            if v > best[0]: best = (v, tau, kk)
    _, tau, kk = best; ki = np.where(c(TAB["test"]) >= tau, 0, kk)
    ev = evaluate(TAB["test"], ki, draft=True, upfront=False); ev["tau"], ev["k_retrieve"] = float(tau), K_OPTIONS[kk]
    return {0: ev}

def run_flare_baseline(lam=LAMBDA_MAIN):
    """FLARE short-form adaptation: if any draft token prob < theta, retrieve with the masked draft as query and regenerate."""
    best = (-1e9, None)
    for th in FLARE_THETAS:
        ev = evaluate(TAB["val"], np.zeros(len(TAB["val"]["qids"]), int), kind="flare", trig=TAB["val"]["FL"]["MINP"] < th, lam=lam)
        v = ev["reward"].mean()
        if v > best[0]: best = (v, th)
    th = best[1]; ev = evaluate(TAB["test"], np.zeros(len(TAB["test"]["qids"]), int), kind="flare",
                                trig=TAB["test"]["FL"]["MINP"] < th)
    ev["theta"] = th; return {0: ev}

def run_skr(lam=LAMBDA_MAIN):
    """SKR-kNN (Wang et al., 2023): retrieve if the nearest training questions were helped by retrieval."""
    kk = K_OPTIONS.index(SKR_K); tr = TAB["train"]
    y = (tr["F1"][:, kk] > tr["F1"][:, 0]).astype(float); Etr = RET["train"]["qemb"]
    def decide(split, m):
        sims = RET[split]["qemb"] @ Etr.T; nn_ = np.argsort(-sims, 1)[:, :m]
        return np.where(y[nn_].mean(1) >= 0.5, kk, 0)
    Rva = reward_matrix(TAB["val"], lam)
    m = max(SKR_NEIGHBORS, key=lambda m: Rva[np.arange(len(Rva)), decide("val", m)].mean())
    q = RET["test"]["qemb"][:1]; t0 = time.perf_counter()
    for _ in range(200): np.argsort(-(q @ Etr.T), 1)[:, :m]
    knn_ms = (time.perf_counter() - t0) / 200 * 1000
    ev = evaluate(TAB["test"], decide("test", m), kind="skr", knn_ms=knn_ms); ev["neighbors"] = m
    return {0: ev}

print("Policies defined.")

In [ ]:
t0 = time.time()
MAIN = {}
for k in K_OPTIONS: MAIN[f"Static k={k}"] = run_static(k)
MAIN["Static (val-best F1)"]     = run_static_best("f1")
MAIN["Static (val-best reward)"] = run_static_best("reward")
MAIN["Random"]                   = run_random()
MAIN["Uncertainty threshold"]    = run_threshold()
MAIN["FLARE (short-form)"]       = run_flare_baseline()
MAIN["SKR-kNN"]                  = run_skr()
MAIN["Supervised k-predictor"]   = run_learned("main_supervised", "supervised")
MAIN["REINFORCE (bandit)"]       = run_learned("main_reinforce", "reinforce")
MAIN["RL-AR (PPO, bandit)"]      = run_learned("main_ppo", "ppo")
MAIN["RL-AR w/o draft (ret+query)"] = run_learned("main_ppo_nodraft", "ppo", groups=("ret", "query"))
MAIN["Oracle (upper bound)"]     = run_oracle()
log_stage("policies_main", time.time() - t0)
print(f"Main experiment: {time.time()-t0:.0f}s")

In [ ]:
METRICS = ["f1", "em", "acc", "k", "total_tokens", "final_prompt_tokens", "latency_ms", "reward"]

def per_query_mean(res, m):   # average each query's metric over seeds
    return np.nanmean(np.stack([ev[m] for ev in res.values()]), 0)

def boot_ci(x, n=N_BOOT, seed=0):
    x = x[np.isfinite(x)]; rng = np.random.RandomState(seed)
    mu = x[rng.randint(len(x), size=(n, len(x)))].mean(1); return np.percentile(mu, [2.5, 97.5])

def paired_test(a, b, n=N_BOOT, seed=0):
    m = np.isfinite(a) & np.isfinite(b); d = a[m] - b[m]; rng = np.random.RandomState(seed)
    bm = d[rng.randint(len(d), size=(n, len(d)))].mean(1)
    p_boot = min(1.0, 2 * min((bm <= 0).mean(), (bm >= 0).mean()))
    try: p_w = sps.wilcoxon(d, zero_method="zsplit").pvalue if np.any(d != 0) else 1.0
    except ValueError: p_w = 1.0
    return float(d.mean()), np.percentile(bm, [2.5, 97.5]), p_boot, float(p_w)

def holm(pvals):
    idx = np.argsort(pvals); m = len(pvals); adj = np.empty(m); run = 0.0
    for r, i in enumerate(idx): run = max(run, min(1.0, (m - r) * pvals[i])); adj[i] = run
    return adj

def results_table(RES, ref_name, metrics=METRICS):
    rows = []
    for name, res in RES.items():
        row = {"Method": name, "seeds": len(res)}
        for m in metrics:
            if m not in next(iter(res.values())): continue
            seed_means = np.array([np.nanmean(ev[m]) for ev in res.values()])
            row[m] = seed_means.mean(); row[m + "_sd"] = seed_means.std(ddof=1) if len(seed_means) > 1 else 0.0
            if m in ("f1", "em"):
                lo, hi = boot_ci(per_query_mean(res, m)); row[m + "_ci"] = f"[{lo:.3f}, {hi:.3f}]"
        rows.append(row)
    df = pd.DataFrame(rows).set_index("Method")
    ref = RES[ref_name]; tests = []
    for name, res in RES.items():
        if name == ref_name or name.startswith("Oracle"): continue
        for m in ["f1", "total_tokens", "latency_ms"]:
            if m not in next(iter(res.values())): continue
            d, ci, pb, pw = paired_test(per_query_mean(res, m), per_query_mean(ref, m))
            tests.append(dict(Method=name, metric=m, delta=d, ci_lo=ci[0], ci_hi=ci[1], p_boot=pb, p_wilcoxon=pw))
    tdf = pd.DataFrame(tests)
    if len(tdf):
        for m in tdf.metric.unique():
            sel = tdf.metric == m; tdf.loc[sel, "p_holm"] = holm(tdf.loc[sel, "p_wilcoxon"].values)
    return df, tdf

REF = "Static (val-best F1)"
main_df, main_tests = results_table(MAIN, REF)
main_df.to_csv(f"{RUN_DIR}/tables/main.csv"); main_tests.to_csv(f"{RUN_DIR}/tables/main_tests_vs_{REF.split()[0]}.csv", index=False)
cols = ["seeds", "f1", "f1_sd", "f1_ci", "em", "em_sd", "acc", "k", "total_tokens", "final_prompt_tokens", "latency_ms", "latency_ms_sd", "reward"]
display(main_df[[c for c in cols if c in main_df]].round(3))
print(f"Paired tests vs {REF} (delta = method - ref; Holm-adjusted Wilcoxon p within each metric):")
display(main_tests.round(4))
_, sup_tests = results_table({k: MAIN[k] for k in ["Supervised k-predictor", "REINFORCE (bandit)", "RL-AR (PPO, bandit)", "RL-AR w/o draft (ret+query)"]}, "Supervised k-predictor")
print("PPO/REINFORCE vs supervised k-predictor:"); display(sup_tests.round(4))

def to_latex(df, cols, path, caption):
    t = df[[c for c in cols if c in df]].copy()
    for m in ["f1", "em", "latency_ms", "total_tokens"]:
        if m in t and m + "_sd" in t:
            fmt = "{:.3f} $\\pm$ {:.3f}" if m in ("f1", "em") else "{:.0f} $\\pm$ {:.0f}"
            t[m] = [fmt.format(a, b) for a, b in zip(t[m], t[m + "_sd"])]; t = t.drop(columns=m + "_sd")
    open(path, "w").write(t.to_latex(escape=False, float_format="%.3f", caption=caption))
to_latex(main_df, ["f1", "f1_sd", "em", "em_sd", "k", "total_tokens", "total_tokens_sd", "latency_ms", "latency_ms_sd"],
         f"{RUN_DIR}/tables/main.tex", f"{DATASET_KEY.upper()}, {MODEL_KEY}: quality and end-to-end cost (mean $\\pm$ SD over seeds).")

# latency component breakdown (R1)
comp = pd.DataFrame({c: [np.nanmean(LAT[c]), np.nanmedian(LAT[c])] for c in LAT}, index=["mean_ms", "median_ms"]).T.round(1)
comp.to_csv(f"{RUN_DIR}/tables/latency_components.csv"); display(comp)

In [ ]:
ABL_SPECS = {
    "Full (unc+ret+query)":      dict(),
    "w/o uncertainty":           dict(groups=("ret", "query")),
    "w/o retrieval scores":      dict(groups=("unc", "query")),
    "w/o query features":        dict(groups=("unc", "ret")),
    "uncertainty only":          dict(groups=("unc",)),
    "retrieval scores only":     dict(groups=("ret",)),
    "query features only":       dict(groups=("query",)),
    "w/o EM term":               dict(w_em=0.0),
    "w/o cost penalty (λ=0)":    dict(lam=0.0),
    "original shaped reward":    dict(reward_kind="shaped"),
    "actions {0,2}":             dict(actions=[0, 2]),
    "actions {0,3}":             dict(actions=[0, 3]),
    "actions {0,1,2}":           dict(actions=[0, 1, 2]),
}
t0 = time.time(); ABL = {}
for name, spec in ABL_SPECS.items():
    slug = "abl_" + re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    ABL[name] = MAIN["RL-AR (PPO, bandit)"] if name.startswith("Full") else run_learned(slug, "ppo", **spec)
ABL["Algorithm: REINFORCE"] = MAIN["REINFORCE (bandit)"]; ABL["Algorithm: supervised"] = MAIN["Supervised k-predictor"]
log_stage("ablations", time.time() - t0)
abl_df, abl_tests = results_table(ABL, "Full (unc+ret+query)")
abl_df.to_csv(f"{RUN_DIR}/tables/ablation.csv"); abl_tests.to_csv(f"{RUN_DIR}/tables/ablation_tests.csv", index=False)
display(abl_df[[c for c in ["f1", "f1_sd", "em", "k", "total_tokens", "latency_ms", "reward", "reward_sd"] if c in abl_df]].round(3))
display(abl_tests.round(4))
# Note: 'original shaped reward' and 'w/o EM'/'λ=0' change the training objective, so compare them on F1/tokens/latency; 'reward' is always the main clean reward.

In [ ]:
t0 = time.time(); FRONT = []
for lam in LAMBDA_GRID:
    tag = str(lam).replace(".", "p")
    for label, res in [("RL-AR (PPO)", run_learned(f"lam_{tag}_ppo", "ppo", lam=lam)),
                       ("Supervised", run_learned(f"lam_{tag}_sup", "supervised", lam=lam)),
                       ("Threshold", run_threshold(lam)),
                       ("Oracle", run_oracle(lam))]:
        s = pd.DataFrame([summarize(ev) for ev in res.values()]).drop(columns="k_dist").mean()
        FRONT.append(dict(method=label, lam=lam, **s.to_dict()))
for k in K_OPTIONS:
    FRONT.append(dict(method="Static", lam=np.nan, **{m: v for m, v in summarize(MAIN[f"Static k={k}"][0]).items() if m != "k_dist"}))
for nm in ["FLARE (short-form)", "SKR-kNN"]:
    FRONT.append(dict(method=nm, lam=np.nan, **{m: v for m, v in summarize(MAIN[nm][0]).items() if m != "k_dist"}))
front = pd.DataFrame(FRONT); front.to_csv(f"{RUN_DIR}/tables/frontier.csv", index=False)
log_stage("frontier", time.time() - t0)

def pareto_mask(cost, qual):
    order = np.argsort(cost); keep = np.zeros(len(cost), bool); best = -np.inf
    for i in order:
        if qual[i] > best: keep[i] = True; best = qual[i]
    return keep

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, cm, lab in [(axes[0], "total_tokens", "Total tokens processed per query"), (axes[1], "latency_ms", "End-to-end latency (ms)")]:
    for meth, mk in [("RL-AR (PPO)", "o-"), ("Supervised", "s--"), ("Threshold", "^:"), ("Oracle", "x-")]:
        d = front[front.method == meth].sort_values(cm); ax.plot(d[cm], d["f1"], mk, label=meth, alpha=0.9)
    d = front[front.method == "Static"]; ax.scatter(d[cm], d["f1"], marker="D", c="k", label="Static k=0..3", zorder=5)
    for _, r in d.iterrows(): ax.annotate(f"k={int(round(r['k']))}", (r[cm], r["f1"]), textcoords="offset points", xytext=(5, -10), fontsize=8)
    for nm, mk in [("FLARE (short-form)", "P"), ("SKR-kNN", "*")]:
        d = front[front.method == nm]; ax.scatter(d[cm], d["f1"], marker=mk, s=90, label=nm, zorder=5)
    ax.set_xlabel(lab); ax.set_ylabel("F1"); ax.grid(alpha=0.3)
axes[0].legend(fontsize=8); fig.suptitle(f"{DATASET_KEY.upper()} | {MODEL_KEY}: quality vs cost (λ sweep)")
fig.tight_layout(); fig.savefig(f"{RUN_DIR}/figs/frontier.png", dpi=200); plt.show()

# training curves with seed bands (R2)
fig, ax = plt.subplots(figsize=(6, 4))
for nm, key in [("PPO", "RL-AR (PPO, bandit)"), ("REINFORCE", "REINFORCE (bandit)")]:
    logs = [pd.DataFrame(ev["train_log"]) for ev in MAIN[key].values()]
    xs = logs[0]["samples"].values; ys = np.stack([l["val_reward"].values[:len(xs)] for l in logs])
    ax.plot(xs, ys.mean(0), label=nm); ax.fill_between(xs, ys.mean(0) - ys.std(0), ys.mean(0) + ys.std(0), alpha=0.2)
ax.set_xlabel("Training samples (query, action) pairs"); ax.set_ylabel("Validation reward (greedy)"); ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig(f"{RUN_DIR}/figs/training_curves.png", dpi=200); plt.show()

In [ ]:
display(pd.DataFrame(RETR_METRICS).T.round(3))
pd.DataFrame(RETR_METRICS).T.to_csv(f"{RUN_DIR}/tables/retrieval_metrics.csv")

te = TAB["test"]; ppo0 = MAIN["RL-AR (PPO, bandit)"][SEEDS[0]]
perk = pd.DataFrame({f"k={k}": dict(F1=te["F1"][:, i].mean(), EM=te["EM"][:, i].mean(), ACC=te["ACC"][:, i].mean(),
                                    prompt_tokens=te["NIN"][:, i].mean()) for i, k in enumerate(K_OPTIONS)}).T
display(perk.round(3)); perk.to_csv(f"{RUN_DIR}/tables/per_k_quality.csv")

qt = pd.DataFrame(dict(qtype=te["qtype"], f1_ppo=per_query_mean(MAIN["RL-AR (PPO, bandit)"], "f1"),
                       k_ppo=per_query_mean(MAIN["RL-AR (PPO, bandit)"], "k"),
                       f1_best_static=MAIN[REF][0]["f1"], f1_k0=te["F1"][:, 0], s1=te["X"][:, FEATURE_NAMES.index("s1")]))
qt_tab = qt.groupby("qtype").agg(n=("f1_ppo", "size"), f1_ppo=("f1_ppo", "mean"), f1_static=("f1_best_static", "mean"),
                                 f1_no_retrieval=("f1_k0", "mean"), avg_k_ppo=("k_ppo", "mean"), mean_top1_sim=("s1", "mean"))
display(qt_tab.round(3)); qt_tab.to_csv(f"{RUN_DIR}/tables/query_type.csv")

# Does the policy's chosen k track retrieval usefulness? (mechanism evidence for R2)
helped = te["F1"][:, 1:].max(1) > te["F1"][:, 0]
print("P(policy retrieves | retrieval helps)  :", round(float((ppo0["k"][helped] > 0).mean()), 3))
print("P(policy retrieves | retrieval doesn't):", round(float((ppo0["k"][~helped] > 0).mean()), 3))

# failure cases: policy skipped retrieval although some k>0 would have answered
fail = [dict(question=te["questions"][j], gold=" | ".join(te["golds"][j][:3]), chosen_k=int(ppo0["k"][j]),
             **{f"ans_k{k}": te["ANS"][j][i] for i, k in enumerate(K_OPTIONS)},
             **{f"f1_k{k}": round(te["F1"][j, i], 2) for i, k in enumerate(K_OPTIONS)})
        for j in range(len(te["qids"])) if ppo0["k"][j] == 0 and te["F1"][j, 0] == 0 and te["F1"][j, 1:].max() > 0.5]
pd.DataFrame(fail).to_csv(f"{RUN_DIR}/tables/failure_cases_skipped_retrieval.csv", index=False)
print(f"{len(fail)} skipped-retrieval failures saved"); display(pd.DataFrame(fail).head(8))

In [ ]:
OTHER = "nq" if DATASET_KEY == "squad" else "squad"
other_dir = f"{OUT_ROOT}/runs/{OTHER}/{MODEL_KEY}"
if all(os.path.exists(f"{other_dir}/tables_{s}.pkl") for s in ["train", "val", "test"]):
    OT = {s: load_pkl(f"{other_dir}/tables_{s}.pkl") for s in ["train", "val", "test"]}
    o_mu, o_sd = OT["train"]["X"].mean(0), OT["train"]["X"].std(0) + 1e-6
    o_tref = float(np.mean(OT["train"]["NIN"][:, -1] - OT["train"]["NIN"][:, 0]))
    rows = []
    for s in SEEDS:
        Xtr = feats(OT["train"], mu=o_mu, sd=o_sd); Xva = feats(OT["val"], mu=o_mu, sd=o_sd)
        Rtr, Rva = reward_matrix(OT["train"], tref=o_tref), reward_matrix(OT["val"], tref=o_tref)
        actor, _ = train_ppo(Xtr, Rtr, Xva, Rva, s)
        ki = greedy(actor, feats(TAB["test"], mu=o_mu, sd=o_sd))
        ev = evaluate(TAB["test"], ki, draft=True, upfront=True, policy_ms=policy_latency_ms(actor, Xtr.shape[1]))
        rows.append(dict(seed=s, **{m: v for m, v in summarize(ev).items() if m != "k_dist"}))
    xfer = pd.DataFrame(rows).drop(columns="seed").agg(["mean", "std"]).T
    xfer["in_domain_mean"] = [np.mean([np.nanmean(ev[m]) for ev in MAIN["RL-AR (PPO, bandit)"].values()]) if m in MAIN["RL-AR (PPO, bandit)"][SEEDS[0]] else np.nan for m in xfer.index]
    print(f"PPO trained on {OTHER.upper()} (frozen), evaluated on {DATASET_KEY.upper()} test:"); display(xfer.round(3))
    xfer.to_csv(f"{RUN_DIR}/tables/transfer_from_{OTHER}.csv")
else:
    print(f"Transfer skipped: run {OTHER} for {MODEL_KEY} through section 6 first.")

In [ ]:
import transformers

transformers.logging.set_verbosity_error()

In [ ]:
SR_PATH = f"{OUT_ROOT}/runs/{DATASET_KEY}/selfrag_test.jsonl"
def run_selfrag():
    items = SPLITS["test"]; done = done_ids(SR_PATH); todo = [j for j, it in enumerate(items) if it["qid"] not in done]
    if not todo: return
    sha = pin("model", SELFRAG_REPO)
    tok = AutoTokenizer.from_pretrained(SELFRAG_REPO, revision=sha)
    model = AutoModelForCausalLM.from_pretrained(SELFRAG_REPO, revision=sha, device_map={"": 0}, torch_dtype=torch.float16,
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                               bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)).eval()
    model.generation_config.max_length = None
    tid = lambda t: tok.convert_tokens_to_ids(t)
    RET_T, NORET_T = tid("[Retrieval]"), tid("[No Retrieval]")
    REL = [tid("[Relevant]"), tid("[Irrelevant]")]
    SUP = [tid("[Fully supported]"), tid("[Partially supported]"), tid("[No support / Contradictory]")]
    USE = [tid(f"[Utility:{i}]") for i in range(1, 6)]
    assert tok.unk_token_id not in [RET_T, NORET_T] + REL + SUP + USE, "Self-RAG special tokens not found"
    R = get_retriever()
    def sr_clean(t):
        t = re.sub(r"<paragraph>.*?</paragraph>", "", t, flags=re.S); t = re.sub(r"\[[^\]]*\]", "", t)
        return clean_answer(t.replace("</s>", ""))
    @torch.inference_mode()
    def gen(prompt):
        enc = tok(prompt, return_tensors="pt").to(model.device)
        out = model.generate(**enc, max_new_tokens=48, do_sample=False, num_beams=1, temperature=None, top_p=None,
                             pad_token_id=tok.eos_token_id, return_dict_in_generate=True, output_scores=True)
        ids = out.sequences[0, enc["input_ids"].shape[1]:].tolist()
        lps = [torch.log_softmax(s[0].float(), -1) for s in out.scores]
        return ids, lps, int(enc["input_ids"].shape[1])
    def grp_prob(ids, lps, group, first_step=False):
        pos = 0 if first_step else next((t for t, i in enumerate(ids) if i in group), None)
        if pos is None or pos >= len(lps): return None
        p = torch.stack([lps[pos][g].exp() for g in group]); return (p / p.sum()).tolist()
    buf = []
    for j in tqdm(todo, desc="Self-RAG"):
        it = items[j]; q = it["question"]; base = f"### Instruction:\n{q}\n\n### Response:\n"
        if torch.cuda.is_available(): torch.cuda.synchronize()
        t0 = time.perf_counter(); tokens = 0
        with torch.inference_mode():
            enc = tok(base, return_tensors="pt").to(model.device); lg = torch.log_softmax(model(**enc).logits[0, -1].float(), -1)
        tokens += enc["input_ids"].shape[1]
        pr = lg[RET_T].exp() / (lg[RET_T].exp() + lg[NORET_T].exp()); retrieve = bool(pr > SELFRAG_THRESHOLD)
        if not retrieve:
            ids, lps, nin = gen(base + "[No Retrieval]"); tokens += nin + len(ids); ans = sr_clean(tok.decode(ids, skip_special_tokens=False))
        else:
            qe = R.encode_queries([q], bs=1); _, ix = R.search(qe, SELFRAG_NDOCS); best = (-1e9, "")
            for i in ix[0]:
                p = PASSAGES[i]
                ids, lps, nin = gen(base + f"[Retrieval]<paragraph>{p['title']}\n{p['text']}</paragraph>"); tokens += nin + len(ids)
                rel = grp_prob(ids, lps, REL, first_step=True); sup = grp_prob(ids, lps, SUP); use = grp_prob(ids, lps, USE)
                score = SELFRAG_W["rel"] * (rel[0] if rel else 0) + SELFRAG_W["sup"] * ((sup[0] + 0.5 * sup[1]) if sup else 0) \
                        + SELFRAG_W["use"] * (sum(w * x for w, x in zip([-1, -0.5, 0, 0.5, 1], use)) if use else 0)
                if score > best[0]: best = (score, sr_clean(tok.decode(ids, skip_special_tokens=False)))
            ans = best[1]
        if torch.cuda.is_available(): torch.cuda.synchronize()
        buf.append(dict(qid=it["qid"], answer=ans, retrieved=retrieve, p_retrieve=float(pr), total_tokens=int(tokens),
                        latency_ms=(time.perf_counter() - t0) * 1000, f1=f1_score(ans, it["answers"]),
                        em=em_score(ans, it["answers"]), acc=acc_score(ans, it["answers"])))
        if len(buf) >= 10: append_jsonl(SR_PATH, buf); buf = []
    if buf: append_jsonl(SR_PATH, buf)
    del model; gc.collect(); torch.cuda.empty_cache()

if RUN_SELFRAG:
    t0 = time.time(); run_selfrag(); log_stage("selfrag", time.time() - t0)
if os.path.exists(SR_PATH):
    sr = pd.DataFrame(read_jsonl(SR_PATH))
    print(f"Self-RAG (official 7B, reference): F1={sr.f1.mean():.3f} EM={sr.em.mean():.3f} ACC={sr.acc.mean():.3f} "
          f"retrieval rate={sr.retrieved.mean():.2f} tokens={sr.total_tokens.mean():.0f} latency={sr.latency_ms.mean():.0f} ms")
    lo, hi = boot_ci(sr.f1.values); print(f"  F1 95% CI [{lo:.3f}, {hi:.3f}]")

In [ ]:
import transformers, sentence_transformers, datasets as hfds
try: import bitsandbytes as bnb; bnb_v = bnb.__version__
except Exception: bnb_v = None
manifest = dict(
    model_key=MODEL_KEY, model_repo=MODELS[MODEL_KEY]["repo"], model_revision=pin("model", MODELS[MODEL_KEY]["repo"]),
    quantization=("bitsandbytes NF4, double quant, fp16 compute" if MODELS[MODEL_KEY]["quant"] else "none (fp16 weights)"),
    embedder=EMBED_REPO, embedder_revision=pin("model", EMBED_REPO), embed_dim=int(RET["train"]["qemb"].shape[1]),
    faiss_index="IndexFlatIP over L2-normalised embeddings (exact cosine)", query_prefix=BGE_QUERY_PREFIX,
    dataset=DATASET_KEY, data_manifest=load_json(f"{DATA_DIR}/data_manifest.json"),
    split_sizes={s: len(v) for s, v in SPLITS.items()}, corpus_size=len(PASSAGES),
    decoding=dict(greedy=True, max_new_tokens=MAX_NEW_TOKENS, prompt="chat template, system + user, see prompt_example_k2.txt"),
    k_options=K_OPTIONS, reward=dict(form="F1 + W_EM*EM - lambda*ctx_tokens/T_ref", W_EM=W_EM, lambda_main=LAMBDA_MAIN, T_ref=T_REF),
    ppo=PPO_CFG, reinforce=REINFORCE_CFG, supervised=SUP_CFG, policy_hidden=POLICY_HIDDEN, seeds=SEEDS,
    checkpoint_selection="best greedy validation reward", flare=dict(k=FLARE_K, mask_prob=FLARE_MASK_PROB, thetas=FLARE_THETAS),
    skr=dict(k=SKR_K, neighbors=SKR_NEIGHBORS), stage_stats=load_json(STATS_PATH, {}),
    versions=dict(python=platform.python_version(), torch=torch.__version__, transformers=transformers.__version__,
                  sentence_transformers=sentence_transformers.__version__, datasets=hfds.__version__,
                  faiss=faiss.__version__, bitsandbytes=bnb_v, cuda=torch.version.cuda),
    gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
save_json(manifest, f"{RUN_DIR}/manifest.json")
print(json.dumps(manifest, indent=1)[:4000])